# Loading

In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time
from scipy import sparse
from tqdm import tqdm
import glob
from sklearn.neighbors import NearestNeighbors

In [ ]:
adata = sc.read_h5ad("fetal_brain_data_ChP/cerebellum.h5ad")
adata

In [ ]:
import json
with open('Output/GRN_GPU_output/cerebellum/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)
grn_dict.keys()

# Celltype speicifc TF expression

In [ ]:
TF_list = []
for TF in grn_dict.keys():
    TF_list.append(TF.split('(')[0])
adata_gene = adata[:,TF_list]
adata_gene

In [ ]:
adata_gene.layers["counts"] = adata_gene.X.copy()
adata_gene.obs_names_make_unique()
sc.pp.normalize_total(adata_gene)
sc.pp.log1p(adata_gene)
sc.pp.scale(adata_gene)

In [ ]:
sc.tl.dendrogram(adata_gene,'dmt_leiden_anno',use_rep='latent_embeddings_all_spatial_pretrain')
sc.tl.rank_genes_groups(adata_gene, 'dmt_leiden_anno', use_rep='latent_embeddings_all_spatial_pretrain',
                        method='wilcoxon',use_raw=False,key_added='leiden_wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_gene,groupby='dmt_leiden_anno',
                                cmap='Spectral_r',key='leiden_wilcoxon',
                                standard_scale='var',n_genes=3,dendrogram=True)

In [ ]:
n_top = 10
top3_genes_dict = {name: adata_gene.uns['leiden_wilcoxon']['names'][name][:n_top] for 
                   name in adata_gene.uns['leiden_wilcoxon']['names'].dtype.names}
top3_genes_dict

In [ ]:
s = pd.Series(top3_genes_dict)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['celltype', 'tf']
df.to_csv("Output/Cere/celltype_specific_tf.csv")
df

In [ ]:
sc.pl.matrixplot(
    adata_gene,
    top3_genes_dict,
    "dmt_leiden_anno",
    dendrogram=True,
    colorbar_title="mean z-score",
    standard_scale = 'var',
    vmin=-1,
    vmax=1,
    cmap="RdBu_r",
)

# h5ad enhancement for AUCell

In [ ]:
from enhance_h5ad import enhance_h5ad_neighbors_mem_efficient

In [ ]:
adata.X.max()

In [ ]:
adata.obs_names = adata.obs_names + adata.obs['slice_code'].tolist()
adata = enhance_h5ad_neighbors_mem_efficient(adata,
                       use_rep='latent_embeddings_all_single_pretrain',
                       annotation='dmt_leiden_anno',temp_path='Process_Data/specific_region/cerebellum_tmp',
                       num_top=50)

In [ ]:
adata.write_h5ad("Process_Data/specific_region/enhanced_cerebellum.h5ad",compression='gzip')

# AUCell calculation

In [ ]:
import pandas as pd
import anndata
from tqdm.auto import tqdm
from typing import Dict, List, Optional
import omicverse as ov
import os
import warnings

def compute_and_concat_adata_by_group(
    adata: anndata.AnnData,
    group_key: str,
    pathway_dict: Dict[str, List[str]],
    output_dir: str,  
    num_workers: int = 30
) -> anndata.AnnData:  

    per_group_results_adatas = []
    unique_groups = list(adata.obs[group_key].cat.categories)

    for g in unique_groups:
        sub_adata = adata[adata.obs[group_key] == g].copy()
        if sub_adata.n_obs == 0:
            continue
            
        sub_adata_processed: anndata.AnnData = ov.single.pathway_aucell_enrichment(
                sub_adata,
                pathways_dict=pathway_dict,
                num_workers=num_workers
            )

        safe_g_name = str(g).replace('/', '_').replace(' ', '_')
        output_filename = os.path.join(output_dir, f"auc_anndata_group_{safe_g_name}.h5ad")

        sub_adata_processed.write_h5ad(output_filename, compression='gzip')

        per_group_results_adatas.append(sub_adata_processed)

    
    adata_all = anndata.concat(per_group_results_adatas, axis=0)

    original_cell_order = adata.obs_names
    cells_in_result = adata_all.obs_names
    final_cell_order = [cell for cell in original_cell_order if cell in cells_in_result]
    
    final_adata = adata_all[final_cell_order].copy()
    
    print("AUCell 计算、保存和 AnnData 拼接完成。")
    return final_adata

In [ ]:
adata_enhance = sc.read_h5ad("Process_Data/specific_region/enhanced_cerebellum.h5ad")
adata_enhance

In [ ]:
import json
with open('Output/GRN_GPU_output/cerebellum/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
auc_mtx = compute_and_concat_adata_by_group(adata_enhance,'dmt_leiden_anno',grn_dict,
                              output_dir='Process_Data/specific_region/auc_cerebellum_adata',num_workers=3)

In [ ]:
auc_mtx.write_h5ad("Process_Data/specific_region/auc_cerebellum.h5ad",compression='gzip')

In [ ]:
files = sorted(glob.glob(os.path.join('Process_Data/specific_region/auc_cerebellum_adata', "*.h5ad")))
adatas = []
for f in tqdm(files):
    print(f'process {f}')
    auc_adata = sc.read_h5ad(f)
    obs_adata = sc.read_h5ad(os.path.join('Process_Data/specific_region/cerebellum_tmp','enhanced_group_'+'_'.join(f.split('/')[-1].split('_')[3:6])))
    auc_adata.obs = obs_adata[auc_adata.obs_names,:].obs
    auc_adata.obsm = obs_adata[auc_adata.obs_names,:].obsm
    auc_adata.write_h5ad(f, compression='gzip')
    #adatas.append(auc_adata)
#combined_adata = ad.concat(adatas, axis=0, join='outer', merge='same')

In [ ]:
files = sorted(glob.glob(os.path.join('Process_Data/specific_region/auc_cerebellum_adata', "*.h5ad")))
adatas = []
for f in tqdm(files):
    print(f'process {f}')
    auc_adata = sc.read_h5ad(f)
    adatas.append(auc_adata)
combined_adata = ad.concat(adatas, axis=0, join='outer', merge='same')
combined_adata.write_h5ad("Process_Data/specific_region/auc_cerebellum.h5ad",
                            compression="gzip")

# AUC visualization

In [ ]:
auc_mtx = sc.read_h5ad("Process_Data/specific_region/auc_cerebellum.h5ad")

In [ ]:
sc.tl.dendrogram(auc_mtx,'dmt_leiden_anno',use_rep='latent_embeddings_all_spatial_pretrain')
sc.tl.rank_genes_groups(auc_mtx, 'dmt_leiden_anno', use_rep='latent_embeddings_all_spatial_pretrain',
                        method='wilcoxon',use_raw=False,key_added='leiden_wilcoxon')
sc.pl.rank_genes_groups_dotplot(auc_mtx,groupby='dmt_leiden_anno',
                                cmap='Spectral_r',key='leiden_wilcoxon',
                                standard_scale='var',n_genes=3,dendrogram=True)

In [ ]:
n_top = 10
top3_genes_dict = {name: adata_gene.uns['leiden_wilcoxon']['names'][name][:n_top]+'(+)' for 
                   name in adata_gene.uns['leiden_wilcoxon']['names'].dtype.names}
s = pd.Series(top3_genes_dict)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['celltype', 'grn']
df.to_csv("Output/Cere/celltype_specific_grn.csv")
df